# Scientific Reports successor — Step 2 bootstrap

This notebook mounts Google Drive, installs the frozen parent and successor packages, validates the parent-source chain, writes a Step 2 bootstrap manifest, and stops. **No scientific calculations are authorized or executed in this notebook.**

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPOSITORY = "https://github.com/khalid-saqr/picoNewton.git"
BRANCH = "successor/scirep-waveform-susceptibility"
COLAB_REPO_ROOT = Path("/content/picoNewton")
DRIVE_SUBDIR = "MyDrive/picoNewton_susceptibility"
IN_COLAB = "google.colab" in sys.modules
print({"in_colab": IN_COLAB, "branch": BRANCH})

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:
    print("Local execution: Google Drive mount skipped.")

In [ ]:
if IN_COLAB:
    if not (COLAB_REPO_ROOT / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(COLAB_REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    REPO_ROOT = COLAB_REPO_ROOT
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    REPO_ROOT = next((root for root in candidates if (root / "picoNewton_v3").is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError("Run locally from inside the picoNewton repository.")
print("Repository root:", REPO_ROOT)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT / "picoNewton_v3")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT / "piconewton_susceptibility")], check=True)

In [ ]:
from piconewton_susceptibility.bootstrap import BootstrapConfig, bootstrap_environment

result = bootstrap_environment(
    BootstrapConfig(
        repo_root=REPO_ROOT,
        storage_mode="drive" if IN_COLAB else "local",
        drive_subdir=DRIVE_SUBDIR,
        local_root=REPO_ROOT / "piconewton_susceptibility_outputs",
    )
)
print(json.dumps(result, indent=2, sort_keys=True))

In [ ]:
manifest = result["manifest"]
assert manifest["scientific_calculations_run"] is False
assert manifest["scientific_calculations_authorized"] is False
assert manifest["parent_source_validation_passed"] is True
print("Step 2 bootstrap complete. Step 3 has not started.")

## Stop boundary

The notebook intentionally ends here. The next authorized action is Step 3: parent-model reproduction and all-six-artery continuity verification, after explicit approval.